In [1]:
"""
So sánh & Đánh giá 4 cấu hình mô hình y tế
============================================
Cấu hình so sánh (theo ảnh yêu cầu):

  Cấu hình A: LLM gốc   + Không RAG   → base model, trả lời từ tham số
  Cấu hình B: LLM gốc   + Có RAG      → base model + retrieval context
  Cấu hình C: LLM fine-tuned + Không RAG → LoRA adapter, không context
  Cấu hình D: LLM fine-tuned + Có RAG    → LoRA adapter + retrieval context

Metrics đánh giá:
  - Định lượng : BLEU-4, ROUGE-L, BERTScore (F1)
  - Retrieval  : Recall@5 (config B, D)
  - Human eval : 50 câu (export CSV để reviewer điền điểm)

Cách chạy trên Colab:
  1. Mount Google Drive
  2. Đảm bảo đã chạy xong 02_finetune và 03_build_rag_faiss
  3. Chỉnh PROJECT_DIR / LORA_DIR / TEST_PATH cho đúng
  4. Runtime → T4 GPU
  5. Chạy từng cell (phân cách bằng # %%)
"""

# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Cài thư viện
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 1] Cài thư viện
!pip install -q \
    transformers datasets peft accelerate "bitsandbytes>=0.46.1" \
    sentence-transformers faiss-cpu \
    langchain langchain-community langchain-text-splitters \
    rouge-score bert-score nltk \
    pandas openpyxl tqdm


# %% [CELL 3] Khai báo tham số
import torch

# Đường dẫn (Cần đảm bảo đúng với thư mục trong Drive của bạn)
MODEL_NAME = "vinai/PhoGPT-4B-Chat" # Ví dụ model gốc, thay bằng model bạn chọn
LORA_ADAPTER = "/content/drive/MyDrive/data/outputs/02_finetune/lora_adapter"
TEST_DATA_PATH = "/content/drive/MyDrive/data/test_data.csv"
FAISS_DB_PATH = "/content/drive/MyDrive/data/vector_db/faiss_index"

# Cấu hình thiết bị
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Đang sử dụng thiết bị: {device}")
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Mount Drive
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 2] Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Khai báo cấu hình và Đường dẫn
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 3]
import os

# Thay đổi các đường dẫn này cho khớp với cấu trúc thư mục của bạn
PROJECT_DIR = "/content/drive/MyDrive/data"
LORA_DIR = f"{PROJECT_DIR}/outputs/02_finetune/lora_adapter"
TEST_PATH = f"{PROJECT_DIR}/data/test_data.csv"
FAISS_DB_PATH = f"{PROJECT_DIR}/vector_db/faiss_index"

# Kiểm tra sự tồn tại của thư mục
if os.path.exists(PROJECT_DIR):
    print(f"✅ Đã tìm thấy thư mục dự án: {PROJECT_DIR}")
else:
    print(f"❌ Cảnh báo: Không tìm thấy thư mục {PROJECT_DIR}. Vui lòng kiểm tra lại đường dẫn.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
🚀 Đang sử dụng thiết bị: cuda
Mounted at /content/drive
✅ Đ

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Import & Cấu hình đường dẫn
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 3] Import & Config
import gc
import json
import os
import re
import unicodedata
import warnings

warnings.filterwarnings("ignore")

import nltk
import numpy as np
import pandas as pd
import torch
from bert_score import score as bert_score_fn
from datasets import Dataset
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from nltk.translate.bleu_score import SmoothingFunction, corpus_bleu, sentence_bleu
from peft import PeftModel
from rouge_score import rouge_scorer
from tqdm import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
    set_seed,
)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

set_seed(42)

# ─── Đường dẫn — CHỈNH LẠI CHO ĐÚNG VỚI DRIVE CỦA BẠN ─────────────────────
# Thư mục gốc của dự án trên Drive
PROJECT_DIR = "/content/drive/MyDrive/data"

# File test QA  (mỗi phần tử: {"instruction": ..., "output": ...})
TEST_PATH = os.path.join(PROJECT_DIR, "qa", "test_qa.json")

# Knowledge base JSONL
KB_PATH = os.path.join(PROJECT_DIR, "final_knowledge_base.jsonl")

# FAISS index đã build ở bước 03
FAISS_DIR = os.path.join(
    PROJECT_DIR, "embeddings", "faiss_medical_index_enriched"
)

# LoRA adapter đã train ở bước 02
LORA_DIR = os.path.join(
    PROJECT_DIR, "models", "qwen2_5_1_5b_medical_lora"
)

# Thư mục lưu kết quả đánh giá
OUTPUT_DIR = os.path.join(PROJECT_DIR, "evaluation")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── Tham số mô hình ─────────────────────────────────────────────────────────
BASE_MODEL      = "Qwen/Qwen2.5-1.5B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MAX_NEW_TOKENS  = 256          # độ dài tối đa câu trả lời sinh ra
RETRIEVER_TOP_K = 5            # Recall@5
RETRIEVER_FETCH_K    = 30
RETRIEVER_LAMBDA_MULT = 0.7

# ─── Số câu đánh giá (giảm để tiết kiệm thời gian GPU trên Colab Free) ──────
# Đổi thành None để chạy toàn bộ test set
MAX_EVAL_SAMPLES = 50          # = số câu human eval

print("✅ Config OK")
print(f"   PROJECT_DIR : {PROJECT_DIR}")
print(f"   TEST_PATH   : {TEST_PATH}")
print(f"   LORA_DIR    : {LORA_DIR}")
print(f"   OUTPUT_DIR  : {OUTPUT_DIR}")

✅ Config OK
   PROJECT_DIR : /content/drive/MyDrive/data
   TEST_PATH   : /content/drive/MyDrive/data/qa/test_qa.json
   LORA_DIR    : /content/drive/MyDrive/data/models/qwen2_5_1_5b_medical_lora
   OUTPUT_DIR  : /content/drive/MyDrive/data/evaluation


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Load test data
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 4] Load test data
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if MAX_EVAL_SAMPLES:
    test_data = test_data[:MAX_EVAL_SAMPLES]

print(f"✅ Số câu hỏi test: {len(test_data)}")
print("Ví dụ:", test_data[0])


def get_question(ex):
    q = ex.get("instruction") or ex.get("question", "")
    extra = ex.get("input", "").strip()
    return f"{q}\n\n{extra}" if extra else q


def get_reference(ex):
    return ex.get("output") or ex.get("answer", "")


questions   = [get_question(ex) for ex in test_data]
references  = [get_reference(ex) for ex in test_data]

print(f"   Câu hỏi mẫu : {questions[0][:80]}")
print(f"   Ref mẫu     : {references[0][:80]}")

✅ Số câu hỏi test: 50
Ví dụ: {'instruction': 'Nguồn thông tin chính về Táo bón từ đâu?', 'input': '', 'output': 'Nguồn: Bệnh viện Đa khoa Lãnh Bình Thăng. Tham khảo: https://benhviendakhoalanhbinhthang.vn/giao-duc-suc-khoe/tao-bon-dau-hieu-nguyen-nhan-dieu-tri-hieu-qua-o-nguoi-lon-tre-em-n3388.html'}
   Câu hỏi mẫu : Nguồn thông tin chính về Táo bón từ đâu?
   Ref mẫu     : Nguồn: Bệnh viện Đa khoa Lãnh Bình Thăng. Tham khảo: https://benhviendakhoalanhb


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Utility: metric functions
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 5] Metric functions

def compute_bleu(hypotheses: list[str], references: list[str]) -> float:
    """BLEU-4 corpus-level với smoothing."""
    smoother = SmoothingFunction().method4
    refs_tok  = [[nltk.word_tokenize(r.lower())] for r in references]
    hyps_tok  = [nltk.word_tokenize(h.lower()) for h in hypotheses]
    return corpus_bleu(refs_tok, hyps_tok, smoothing_function=smoother)


def compute_rouge_l(hypotheses: list[str], references: list[str]) -> float:
    """ROUGE-L trung bình."""
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    scores = [
        scorer.score(ref, hyp)["rougeL"].fmeasure
        for hyp, ref in zip(hypotheses, references)
    ]
    return float(np.mean(scores))


def compute_bertscore(
    hypotheses: list[str], references: list[str], lang: str = "vi"
) -> float:
    """BERTScore F1 trung bình (dùng xlm-roberta)."""
    P, R, F1 = bert_score_fn(
        hypotheses,
        references,
        lang=lang,
        model_type="xlm-roberta-base",
        verbose=False,
    )
    return float(F1.mean())


def compute_recall_at_k(
    retrieved_docs_list: list[list[Document]],
    references: list[str],
    k: int = 5,
) -> float:
    """
    Recall@k: tỉ lệ câu hỏi có ít nhất 1 chunk retrieved
    chứa ≥ 1 bigram quan trọng từ reference answer.
    (Proxy metric — không cần ground-truth passage ids)
    """
    hits = 0
    for docs, ref in zip(retrieved_docs_list, references):
        ref_norm = normalize_text(ref)
        # lấy các bigram 3-token từ reference
        ref_words  = ref_norm.split()
        ref_ngrams = set(
            " ".join(ref_words[i : i + 3])
            for i in range(len(ref_words) - 2)
        )
        found = False
        for doc in docs[:k]:
            doc_norm = normalize_text(doc.page_content)
            for ng in ref_ngrams:
                if ng in doc_norm:
                    found = True
                    break
            if found:
                break
        if found:
            hits += 1
    return hits / len(references) if references else 0.0


def normalize_text(text: str) -> str:
    text = text.lower()
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    return text.replace("\u0111", "d")


print("✅ Metric functions OK")

✅ Metric functions OK


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Load tokenizer (dùng chung)
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 6] Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Tokenizer loaded:", BASE_MODEL)


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: Qwen/Qwen2.5-1.5B-Instruct


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Build RAG pipeline (dùng chung cho B và D)
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 7] Build RAG pipeline
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# — Load knowledge base & chunking
raw_docs = []
with open(KB_PATH, "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        raw_docs.append(
            Document(
                page_content=item["page_content"],
                metadata=item.get("metadata", {}),
            )
        )

splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)
print(f"✅ KB: {len(raw_docs)} docs → {len(chunks)} chunks")

# — Embedding & FAISS
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

if os.path.exists(FAISS_DIR):
    vectorstore = FAISS.load_local(
        FAISS_DIR, embedding_model, allow_dangerous_deserialization=True
    )
    print("✅ FAISS index loaded từ:", FAISS_DIR)
else:
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    vectorstore.save_local(FAISS_DIR)
    print("✅ FAISS index mới tạo và lưu tại:", FAISS_DIR)


# ─── Metadata inference (copy từ file 03) ────────────────────────────────────
ALL_DISEASES = sorted(
    {doc.metadata.get("disease", "") for doc in chunks if doc.metadata.get("disease", "")},
    key=lambda d: len(normalize_text(d)),
    reverse=True,
)

DISEASE_SYNONYMS = {
    "viem phoi":        ["pneumonia", "sung phoi", "nhiem trung phoi"],
    "dai thao duong":   ["tieu duong", "diabetes", "diabetes mellitus"],
    "tang huyet ap":    ["cao huyet ap", "hypertension"],
    "gout (gut)":       ["gout", "gut", "thong phong"],
    "viem hong":        ["dau hong", "pharyngitis"],
    "viem da day":      ["dau da day", "gastritis"],
    "tao bon":          ["constipation"],
    "tieu chay":        ["diarrhea"],
    "tram cam":         ["depression"],
    "mat ngu":          ["insomnia"],
    "dau dau":          ["nhuc dau", "headache"],
    "kho tho":          ["shortness of breath", "dyspnea"],
    "ho":               ["cough"],
}

SECTION_INTENTS = {
    "symptoms":           ["trieu chung", "dau hieu", "bieu hien", "nhan biet"],
    "causes":             ["nguyen nhan", "tai sao", "do dau"],
    "treatment":          ["dieu tri", "chua", "thuoc", "xu tri"],
    "when_to_see_doctor": ["khi nao", "di kham", "gap bac si", "cap cuu"],
    "prevention":         ["phong ngua", "phong benh"],
    "definition":         ["la gi", "dinh nghia", "tong quan"],
    "risk_groups":        ["nguy co", "doi tuong"],
    "care_notes":         ["luu y", "cham soc"],
}


def contains_whole_phrase(text: str, phrase: str) -> bool:
    phrase = normalize_text(phrase).strip()
    if not phrase:
        return False
    return re.search(r"(?<!\w)" + re.escape(phrase) + r"(?!\w)", text) is not None


def get_disease_aliases(disease: str) -> list[str]:
    normalized = normalize_text(disease)
    aliases = [normalized] + DISEASE_SYNONYMS.get(normalized, [])
    return sorted(set(aliases), key=len, reverse=True)


def infer_query_metadata(question: str):
    nq = normalize_text(question)
    matched_disease = None
    for d in ALL_DISEASES:
        if any(contains_whole_phrase(nq, a) for a in get_disease_aliases(d)):
            matched_disease = d
            break
    matched_section = None
    for sec, kws in SECTION_INTENTS.items():
        if any(k in nq for k in kws):
            matched_section = sec
            break
    return matched_disease, matched_section


def metadata_boost(doc: Document, md: str | None, ms: str | None) -> float:
    b = 0.0
    if md and doc.metadata.get("disease") == md:
        b += 2.0
    if ms and doc.metadata.get("section") == ms:
        b += 1.0
    if md and "ho hap" in normalize_text(doc.metadata.get("topic", "")) \
           and "phoi" in normalize_text(md):
        b += 0.3
    return b


def retrieve_hybrid(question: str, top_k: int = RETRIEVER_TOP_K) -> list[Document]:
    """Trả về list[Document] top-k sau hybrid rerank."""
    md, ms = infer_query_metadata(question)
    fetch_n = min(RETRIEVER_FETCH_K, len(chunks))
    candidates = vectorstore.similarity_search_with_score(question, k=fetch_n)

    seen, deduped = set(), []
    for doc, dist in candidates:
        key = doc.page_content[:120]
        if key not in seen:
            seen.add(key)
            deduped.append((doc, float(dist)))

    reranked = sorted(
        deduped,
        key=lambda item: item[1] - metadata_boost(item[0], md, ms),
    )
    return [doc for doc, _ in reranked[:top_k]]


def build_context(docs: list[Document]) -> str:
    parts = []
    for idx, doc in enumerate(docs, 1):
        parts.append(
            f"[Tài liệu {idx}]\n"
            f"Bệnh: {doc.metadata.get('disease','')} | Mục: {doc.metadata.get('section','')}\n"
            f"Nội dung: {doc.page_content}"
        )
    return "\n\n".join(parts)


print("✅ RAG pipeline OK")



Device: cuda
✅ KB: 779 docs → 818 chunks


/tmp/ipykernel_1323/2636083541.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS index loaded từ: /content/drive/MyDrive/data/embeddings/faiss_medical_index_enriched
✅ RAG pipeline OK


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Prompt builders
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 8] Prompt builders

SYSTEM_PLAIN = (
    "Bạn là trợ lý y tế phổ thông. "
    "Trả lời ngắn gọn, dễ hiểu, không chẩn đoán thay bác sĩ."
)

SYSTEM_RAG = (
    "Bạn là trợ lý hỏi đáp y tế. Trả lời bằng tiếng Việt.\n\n"
    "QUY TẮC BẮT BUỘC:\n"
    "1. Chỉ dùng thông tin có trong NGỮ CẢNH bên dưới.\n"
    "2. KHÔNG dùng kiến thức bên ngoài NGỮ CẢNH.\n"
    "3. Nếu không có thông tin phù hợp, trả lời: "
    "\"Tôi chưa có đủ thông tin trong tài liệu để trả lời chính xác.\"\n"
    "4. KHÔNG chẩn đoán bệnh. KHÔNG kê đơn thuốc.\n"
    "5. Trả lời ngắn gọn, đúng trọng tâm, dạng văn xuôi hoặc danh sách (-).\n"
    "6. TUYỆT ĐỐI KHÔNG tạo câu hỏi trắc nghiệm, quiz."
)


def build_prompt_no_rag(question: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PLAIN},
        {"role": "user",   "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def build_prompt_rag(question: str, context: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_RAG},
        {"role": "user",   "content": f"NGỮ CẢNH:\n{context}\n\nCÂU HỎI:\n{question}"},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


class BadFormatStopper(StoppingCriteria):
    STOP_PHRASES = ["trắc nghiệm", "chọn đáp án", "a) ", "b) ", "c) ", "d) "]
    def __init__(self, tok):
        self.tok = tok
    def __call__(self, input_ids, scores, **kwargs):
        decoded = self.tok.decode(input_ids[0, -30:]).lower()
        return any(p in decoded for p in self.STOP_PHRASES)


def generate_answer(model, prompt: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stopper = BadFormatStopper(tokenizer)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.15,
            eos_token_id=tokenizer.eos_token_id,
            stopping_criteria=StoppingCriteriaList([stopper]),
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


print("✅ Prompt builders & generate function OK")


✅ Prompt builders & generate function OK


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 — Helper: load model (base hoặc fine-tuned)
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 9] Model loader

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


def load_base_model():
    print("⏳ Loading base model...")
    m = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    m.eval()
    print("✅ Base model loaded")
    return m


def load_finetuned_model():
    print("⏳ Loading fine-tuned model (base + LoRA)...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    ft = PeftModel.from_pretrained(base, LORA_DIR)
    ft.config.use_cache = True
    ft.eval()
    print("✅ Fine-tuned model loaded")
    return ft


def unload_model(model):
    """Giải phóng VRAM trước khi load model tiếp theo."""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("🗑️  Model unloaded, VRAM freed")


# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 — Run evaluation cho 1 config
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 10] Run single config evaluation

def run_config(
    config_name: str,
    model,
    questions: list[str],
    references: list[str],
    use_rag: bool,
) -> dict:
    """
    Chạy inference cho toàn bộ test set với 1 config.
    Trả về dict kết quả gồm answers, metrics.
    """
    print(f"\n{'='*60}")
    print(f"▶ Config {config_name} | RAG={'Yes' if use_rag else 'No'}")
    print(f"{'='*60}")

    answers      = []
    retrieved_docs_list = []   # chỉ dùng khi use_rag=True

    for i, question in enumerate(tqdm(questions, desc=f"Config {config_name}")):
        try:
            if use_rag:
                docs    = retrieve_hybrid(question)
                context = build_context(docs)
                prompt  = build_prompt_rag(question, context)
                retrieved_docs_list.append(docs)
            else:
                prompt = build_prompt_no_rag(question)
                retrieved_docs_list.append([])

            answer = generate_answer(model, prompt)
            answers.append(answer)

        except Exception as e:
            print(f"  ⚠ Lỗi câu {i}: {e}")
            answers.append("")
            retrieved_docs_list.append([])

    # ── Tính metrics ──────────────────────────────────────────────────────
    print(f"\n📊 Đang tính metrics cho config {config_name}...")

    bleu    = compute_bleu(answers, references)
    rouge_l = compute_rouge_l(answers, references)

    print("   BLEU & ROUGE-L ✓")

    bs = compute_bertscore(answers, references)
    print("   BERTScore ✓")

    recall_at_k = (
        compute_recall_at_k(retrieved_docs_list, references, k=RETRIEVER_TOP_K)
        if use_rag
        else None
    )

    metrics = {
        "config":      config_name,
        "use_rag":     use_rag,
        "bleu_4":      round(bleu, 4),
        "rouge_l":     round(rouge_l, 4),
        "bertscore_f1": round(bs, 4),
        "recall_at_5": round(recall_at_k, 4) if recall_at_k is not None else "N/A",
        "n_samples":   len(answers),
    }

    print(f"\n✅ Config {config_name} kết quả:")
    for k, v in metrics.items():
        print(f"   {k:20s}: {v}")

    return {
        "metrics":   metrics,
        "answers":   answers,
        "retrieved": retrieved_docs_list,
    }

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 11 — Chạy Config A: LLM gốc, KHÔNG RAG
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 11] Config A — Base LLM, No RAG
model_base = load_base_model()

result_A = run_config(
    config_name="A (Base, No RAG)",
    model=model_base,
    questions=questions,
    references=references,
    use_rag=False,
)

# Lưu tạm để dùng sau
answers_A = result_A["answers"]
metrics_A = result_A["metrics"]

⏳ Loading base model...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Base model loaded

▶ Config A (Base, No RAG) | RAG=No


...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   BERTScore ✓

✅ Config A (Base, No RAG) kết quả:
   config              : A (Base, No RAG)
   use_rag             : False
   bleu_4              : 0.0248
   rouge_l             : 0.2717
   bertscore_f1        : 0.8353
   recall_at_5         : N/A
   n_samples           : 50


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 12 — Chạy Config B: LLM gốc, CÓ RAG
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 12] Config B — Base LLM + RAG
# (dùng lại model_base đang load, không cần load lại)

result_B = run_config(
    config_name="B (Base, RAG)",
    model=model_base,
    questions=questions,
    references=references,
    use_rag=True,
)

answers_B = result_B["answers"]
metrics_B = result_B["metrics"]

# Giải phóng VRAM trước khi load fine-tuned model
unload_model(model_base)


▶ Config B (Base, RAG) | RAG=Yes


Config B (Base, RAG): 100%|██████████| 50/50 [11:49<00:00, 14.19s/it]



📊 Đang tính metrics cho config B (Base, RAG)...
   BLEU & ROUGE-L ✓


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   BERTScore ✓

✅ Config B (Base, RAG) kết quả:
   config              : B (Base, RAG)
   use_rag             : True
   bleu_4              : 0.1373
   rouge_l             : 0.3993
   bertscore_f1        : 0.8634
   recall_at_5         : 0.84
   n_samples           : 50
🗑️  Model unloaded, VRAM freed


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 13 — Chạy Config C: LLM fine-tuned, KHÔNG RAG
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 13] Config C — Fine-tuned LLM, No RAG
model_ft = load_finetuned_model()

result_C = run_config(
    config_name="C (Fine-tuned, No RAG)",
    model=model_ft,
    questions=questions,
    references=references,
    use_rag=False,
)

answers_C = result_C["answers"]
metrics_C = result_C["metrics"]

⏳ Loading fine-tuned model (base + LoRA)...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Fine-tuned model loaded

▶ Config C (Fine-tuned, No RAG) | RAG=No


Config C (Fine-tuned, No RAG): 100%|██████████| 50/50 [04:00<00:00,  4.81s/it]



📊 Đang tính metrics cho config C (Fine-tuned, No RAG)...
   BLEU & ROUGE-L ✓


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   BERTScore ✓

✅ Config C (Fine-tuned, No RAG) kết quả:
   config              : C (Fine-tuned, No RAG)
   use_rag             : False
   bleu_4              : 0.0475
   rouge_l             : 0.2982
   bertscore_f1        : 0.8469
   recall_at_5         : N/A
   n_samples           : 50


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 14 — Chạy Config D: LLM fine-tuned, CÓ RAG
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 14] Config D — Fine-tuned LLM + RAG
result_D = run_config(
    config_name="D (Fine-tuned, RAG)",
    model=model_ft,
    questions=questions,
    references=references,
    use_rag=True,
)

answers_D = result_D["answers"]
metrics_D = result_D["metrics"]

unload_model(model_ft)


▶ Config D (Fine-tuned, RAG) | RAG=Yes


Config D (Fine-tuned, RAG): 100%|██████████| 50/50 [07:18<00:00,  8.77s/it]



📊 Đang tính metrics cho config D (Fine-tuned, RAG)...
   BLEU & ROUGE-L ✓


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   BERTScore ✓

✅ Config D (Fine-tuned, RAG) kết quả:
   config              : D (Fine-tuned, RAG)
   use_rag             : True
   bleu_4              : 0.2161
   rouge_l             : 0.4167
   bertscore_f1        : 0.8689
   recall_at_5         : 0.84
   n_samples           : 50
🗑️  Model unloaded, VRAM freed


In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 15 — Tổng hợp & In bảng kết quả
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 15] Tổng hợp kết quả

all_metrics = [metrics_A, metrics_B, metrics_C, metrics_D]
df_metrics  = pd.DataFrame(all_metrics)

print("\n" + "═" * 70)
print("BẢNG SO SÁNH 4 CẤU HÌNH")
print("═" * 70)
print(df_metrics.to_string(index=False))
print("═" * 70)

# Xác định config tốt nhất theo từng metric
for metric in ["bleu_4", "rouge_l", "bertscore_f1"]:
    vals   = df_metrics[metric].tolist()
    best_i = int(np.argmax(vals))
    print(
        f"  🏆 {metric:<20s}: tốt nhất = Config "
        f"{df_metrics.iloc[best_i]['config']} ({vals[best_i]:.4f})"
    )

# Recall@5 (chỉ có B và D)
recall_vals = df_metrics[df_metrics["recall_at_5"] != "N/A"]["recall_at_5"].astype(float)
if not recall_vals.empty:
    best_recall = recall_vals.idxmax()
    print(
        f"  🏆 {'recall_at_5':<20s}: tốt nhất = Config "
        f"{df_metrics.loc[best_recall, 'config']} ({recall_vals[best_recall]:.4f})"
    )


══════════════════════════════════════════════════════════════════════
BẢNG SO SÁNH 4 CẤU HÌNH
══════════════════════════════════════════════════════════════════════
                config  use_rag  bleu_4  rouge_l  bertscore_f1 recall_at_5  n_samples
      A (Base, No RAG)    False  0.0248   0.2717        0.8353         N/A         50
         B (Base, RAG)     True  0.1373   0.3993        0.8634        0.84         50
C (Fine-tuned, No RAG)    False  0.0475   0.2982        0.8469         N/A         50
   D (Fine-tuned, RAG)     True  0.2161   0.4167        0.8689        0.84         50
══════════════════════════════════════════════════════════════════════
  🏆 bleu_4              : tốt nhất = Config D (Fine-tuned, RAG) (0.2161)
  🏆 rouge_l             : tốt nhất = Config D (Fine-tuned, RAG) (0.4167)
  🏆 bertscore_f1        : tốt nhất = Config D (Fine-tuned, RAG) (0.8689)
  🏆 recall_at_5         : tốt nhất = Config B (Base, RAG) (0.8400)


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 16 — Lưu kết quả định lượng
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 16] Lưu kết quả định lượng

# CSV tổng hợp
metrics_csv = os.path.join(OUTPUT_DIR, "metrics_summary.csv")
df_metrics.to_csv(metrics_csv, index=False, encoding="utf-8-sig")
print(f"✅ Lưu metrics summary: {metrics_csv}")

# JSON chi tiết
metrics_json = os.path.join(OUTPUT_DIR, "metrics_summary.json")
with open(metrics_json, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, ensure_ascii=False, indent=2)
print(f"✅ Lưu metrics JSON: {metrics_json}")

✅ Lưu metrics summary: /content/drive/MyDrive/data/evaluation/metrics_summary.csv
✅ Lưu metrics JSON: /content/drive/MyDrive/data/evaluation/metrics_summary.json


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 17 — Export file human evaluation (50 câu)
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 17] Export Human Eval CSV

human_rows = []
for i, (q, ref) in enumerate(zip(questions, references)):
    row = {
        "stt":                   i + 1,
        "question":              q,
        "reference_answer":      ref,
        "answer_A_base_norag":   answers_A[i] if i < len(answers_A) else "",
        "answer_B_base_rag":     answers_B[i] if i < len(answers_B) else "",
        "answer_C_ft_norag":     answers_C[i] if i < len(answers_C) else "",
        "answer_D_ft_rag":       answers_D[i] if i < len(answers_D) else "",
        # Cột để reviewer điền — thang điểm 1-5
        "score_A (1-5)":         "",
        "score_B (1-5)":         "",
        "score_C (1-5)":         "",
        "score_D (1-5)":         "",
        "ghi_chu_reviewer":      "",
    }
    human_rows.append(row)

df_human = pd.DataFrame(human_rows)

# Lưu CSV
human_csv = os.path.join(OUTPUT_DIR, "human_eval_50_questions.csv")
df_human.to_csv(human_csv, index=False, encoding="utf-8-sig")
print(f"✅ Human eval CSV: {human_csv}")

# Lưu Excel (dễ nhìn hơn khi reviewer dùng)
human_xlsx = os.path.join(OUTPUT_DIR, "human_eval_50_questions.xlsx")
with pd.ExcelWriter(human_xlsx, engine="openpyxl") as writer:
    df_human.to_excel(writer, index=False, sheet_name="Human Eval")
    ws = writer.sheets["Human Eval"]
    # Điều chỉnh độ rộng cột
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 60)
print(f"✅ Human eval Excel: {human_xlsx}")

✅ Human eval CSV: /content/drive/MyDrive/data/evaluation/human_eval_50_questions.csv
✅ Human eval Excel: /content/drive/MyDrive/data/evaluation/human_eval_50_questions.xlsx


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 18 — Phân tích chi tiết per-sample (optional)
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 18] Per-sample detailed analysis

scorer_rl = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
detailed_rows = []

for i, (q, ref) in enumerate(zip(questions, references)):
    row = {
        "stt":            i + 1,
        "question":       q[:100],
        "reference":      ref[:150],
    }
    for cfg, answers in [
        ("A", answers_A), ("B", answers_B), ("C", answers_C), ("D", answers_D)
    ]:
        ans = answers[i] if i < len(answers) else ""
        row[f"answer_{cfg}"] = ans[:150]
        # ROUGE-L per sample
        row[f"rouge_l_{cfg}"] = round(
            scorer_rl.score(ref, ans)["rougeL"].fmeasure, 4
        )
    detailed_rows.append(row)

df_detailed = pd.DataFrame(detailed_rows)

detailed_csv = os.path.join(OUTPUT_DIR, "per_sample_analysis.csv")
df_detailed.to_csv(detailed_csv, index=False, encoding="utf-8-sig")
print(f"✅ Per-sample analysis: {detailed_csv}")

✅ Per-sample analysis: /content/drive/MyDrive/data/evaluation/per_sample_analysis.csv


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 19 — In ví dụ so sánh 3 câu hỏi
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 19] Ví dụ so sánh

sample_indices = [0, 1, 2]  # Đổi index nếu muốn xem câu khác

for idx in sample_indices:
    print(f"\n{'─'*65}")
    print(f"[Câu {idx+1}] {questions[idx]}")
    print(f"{'─'*65}")
    print(f"📌 Reference   : {references[idx][:200]}")
    print(f"\n🅰 Config A    : {answers_A[idx][:200]}")
    print(f"🅱 Config B    : {answers_B[idx][:200]}")
    print(f"🅲 Config C    : {answers_C[idx][:200]}")
    print(f"🅳 Config D    : {answers_D[idx][:200]}")


─────────────────────────────────────────────────────────────────
[Câu 1] Nguồn thông tin chính về Táo bón từ đâu?
─────────────────────────────────────────────────────────────────
📌 Reference   : Nguồn: Bệnh viện Đa khoa Lãnh Bình Thăng. Tham khảo: https://benhviendakhoalanhbinhthang.vn/giao-duc-suc-khoe/tao-bon-dau-hieu-nguyen-nhan-dieu-tri-hieu-qua-o-nguoi-lon-tre-em-n3388.html

🅰 Config A    : Thông tin về bệnh táo bón thường được tìm kiếm trên các trang web của WHO (World Health Organization), CDC (Centers for Disease Control and Prevention) hoặc các nguồn nghiên cứu khoa học uy tín trong 
🅱 Config B    : Nguyên nguồn thông tin chính về Táo bón từ các tài liệu là:

1. Tài liệu 1: "Bệnh: Táo bón" của Vinmec.
2. Tài liệu 2: "Bệnh: Táo bón" của Bệnh viện Đa khoa Lãnh Bình Thăng.
3. Tài liệu 3: "Bệnh: Táo 
🅲 Config C    : Thông tin chính thức đến từ các cơ quan y tế như WHO và CDC trong thế giới thực, cùng với các nghiên cứu khoa học uy tín.
🅳 Config D    : Sách 1 của nhà xuất bản Vi

In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 20 — In tóm tắt cuối cùng
# ══════════════════════════════════════════════════════════════════════════════
# %% [CELL 20] Final summary

print("\n" + "★" * 70)
print("TỔNG KẾT ĐÁNH GIÁ 4 CẤU HÌNH")
print("★" * 70)
print(f"""
  Cấu hình A: LLM gốc   + Không RAG
  Cấu hình B: LLM gốc   + Có RAG
  Cấu hình C: LLM fine-tuned + Không RAG
  Cấu hình D: LLM fine-tuned + Có RAG  ← kỳ vọng tốt nhất

  Metrics sử dụng:
    • BLEU-4      : đo độ trùng n-gram
    • ROUGE-L     : đo chuỗi con dài nhất chung
    • BERTScore F1: đo semantic similarity
    • Recall@5    : tỉ lệ retrieved chunk có nội dung đúng (B, D)
    • Human eval  : 50 câu — reviewer điền điểm trong file Excel

  Files kết quả lưu tại: {OUTPUT_DIR}
    ├── metrics_summary.csv
    ├── metrics_summary.json
    ├── human_eval_50_questions.csv
    ├── human_eval_50_questions.xlsx   ← gửi reviewer
    └── per_sample_analysis.csv
""")
print("★" * 70)



★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
TỔNG KẾT ĐÁNH GIÁ 4 CẤU HÌNH
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

  Cấu hình A: LLM gốc   + Không RAG
  Cấu hình B: LLM gốc   + Có RAG
  Cấu hình C: LLM fine-tuned + Không RAG
  Cấu hình D: LLM fine-tuned + Có RAG  ← kỳ vọng tốt nhất
 
  Metrics sử dụng:
    • BLEU-4      : đo độ trùng n-gram
    • ROUGE-L     : đo chuỗi con dài nhất chung
    • BERTScore F1: đo semantic similarity
    • Recall@5    : tỉ lệ retrieved chunk có nội dung đúng (B, D)
    • Human eval  : 50 câu — reviewer điền điểm trong file Excel
 
  Files kết quả lưu tại: /content/drive/MyDrive/data/evaluation
    ├── metrics_summary.csv
    ├── metrics_summary.json
    ├── human_eval_50_questions.csv
    ├── human_eval_50_questions.xlsx   ← gửi reviewer
    └── per_sample_analysis.csv

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
